# Build Disruption Features at FAF Zone-Year and OD-Year Level

This notebook builds the next required feature block for FREIGHT-MNet: **annual disruption exposure features** aligned with the supervised FAF OD-year quantile prediction task.

The output is designed for the next model notebook, `DCQHGT_Disruption_ColdOD.ipynb`. It produces:

- a FAF zone-year disruption table;
- an OD-year disruption table aligned one-to-one with the supervised model-ready edge table;
- a model-ready supervised table with disruption columns appended;
- a disruption feature manifest and QA artifacts.

The implementation is intentionally robust to partially processed source folders. If a NOAA, STB, or border source is missing or uses an unexpected schema, the notebook records the issue and creates explicit zero-valued fallback features instead of silently failing. This keeps the downstream D-CQHGT experiments reproducible while making missing-source coverage transparent.

## 1. Kernel preflight

This project has repeatedly hit NumPy ABI and PyG environment issues when Jupyter accidentally uses the base Anaconda kernel. This first cell checks the active Python executable **before importing pandas**. It is expected to run in the FREIGHT-MNet CUDA/PyG environment.

If this cell raises, switch the Jupyter kernel to `Python (freight_mnet_cuda)` or register that kernel using the printed command.

In [1]:
# ---------------------------------------------------------------------
# Kernel preflight before importing pandas or heavy compiled packages
# ---------------------------------------------------------------------

import os
import sys
from pathlib import Path

EXPECTED_PYTHON = Path(r"E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe")
STRICT_KERNEL_CHECK = False  # Set True if you want to force the CUDA/PyG environment.

current_python = Path(sys.executable)
print("Current Python executable:", current_python)
print("Expected project Python:", EXPECTED_PYTHON)

if STRICT_KERNEL_CHECK and current_python.resolve() != EXPECTED_PYTHON.resolve():
    print("\nTo register the expected kernel, run this in Windows CMD or PowerShell:")
    print(f'"{EXPECTED_PYTHON}" -m pip install ipykernel')
    print(f'"{EXPECTED_PYTHON}" -m ipykernel install --user --name freight_mnet_cuda --display-name "Python (freight_mnet_cuda)"')
    raise RuntimeError(
        "The active Jupyter kernel is not the expected FREIGHT-MNet CUDA/PyG environment. "
        "Switch the kernel before continuing."
    )

Current Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Expected project Python: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe


## 2. Imports and global configuration

The notebook uses pandas and NumPy for feature construction. GeoPandas is not required here because all geospatial topology has already been converted into HeteroData and topology feature artifacts. Optional plotting is disabled by default.

In [2]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

from __future__ import annotations

import json
import math
import os
import re
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# Avoid optional acceleration backends that have caused ABI issues in mixed NumPy environments.
pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", os.sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Python: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
NumPy: 2.4.5
Pandas: 2.3.3


## 3. Experiment configuration

The paths follow the current project layout. The default scope is `east_plus_gulf`, matching the main experimental scope used throughout the FREIGHT-MNet experiments.

The notebook is allowed to create fallback zero-valued features if a source folder is missing. This behavior is controlled by `allow_missing_sources`.

In [3]:
# ---------------------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------------------

@dataclass
class DisruptionFeatureConfig:
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "build_disruption_features_od_year_v1"
    seed: int = 42

    # Source folders from the project report and data pipeline.
    noaa_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events")
    stb_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance")
    border_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data\06_border_crossing")
    crosswalk_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf")

    # Processing controls.
    allow_missing_sources: bool = True
    max_noaa_files: Optional[int] = None
    max_stb_files: Optional[int] = None
    max_border_files: Optional[int] = None
    recursive_source_search: bool = True

    # Output controls.
    overwrite: bool = True
    float_output_precision: int = 6

cfg = DisruptionFeatureConfig()
np.random.seed(cfg.seed)

print("Configuration:")
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))

Configuration:
{
  "data_root": "E:\\NetworkOptimization\\pythonProject1\\Data",
  "scope": "east_plus_gulf",
  "run_name": "build_disruption_features_od_year_v1",
  "seed": "42",
  "noaa_root": "E:\\NetworkOptimization\\pythonProject1\\Data\\05_noaa_storm_events",
  "stb_root": "E:\\NetworkOptimization\\pythonProject1\\Data\\04_stb\\rail_service_performance",
  "border_root": "E:\\NetworkOptimization\\pythonProject1\\Data\\06_border_crossing",
  "crosswalk_root": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf",
  "allow_missing_sources": "True",
  "max_noaa_files": "None",
  "max_stb_files": "None",
  "max_border_files": "None",
  "recursive_source_search": "True",
  "overwrite": "True",
  "float_output_precision": "6"
}


## 4. Path management

This cell centralizes all required input and output paths. The output directory is split into a reusable processed-feature directory and an experiment QA directory.

In [4]:
# ---------------------------------------------------------------------
# Path management
# ---------------------------------------------------------------------

@dataclass
class DisruptionPaths:
    data_root: Path
    supervised_path: Path
    manifest_path: Path
    processed_disruption_dir: Path
    experiment_dir: Path
    tables_dir: Path
    zone_year_features_path: Path
    od_year_features_path: Path
    supervised_with_disruption_path: Path
    disruption_manifest_path: Path
    run_config_path: Path
    qa_summary_path: Path
    source_audit_path: Path


def build_paths(config: DisruptionFeatureConfig) -> DisruptionPaths:
    """Build all project paths for the disruption-feature pipeline."""
    processed_dir = config.data_root / "08_processed" / "disruption_features"
    experiment_dir = config.data_root / "10_experiments" / config.run_name / config.scope
    tables_dir = experiment_dir / "tables"

    return DisruptionPaths(
        data_root=config.data_root,
        supervised_path=config.data_root / "08_processed" / "model_ready" / f"freight_mnet_supervised_edges_2018_2024_{config.scope}.parquet",
        manifest_path=config.data_root / "08_processed" / "model_ready" / "_metadata" / "freight_mnet_supervised_feature_manifest.json",
        processed_disruption_dir=processed_dir,
        experiment_dir=experiment_dir,
        tables_dir=tables_dir,
        zone_year_features_path=processed_dir / f"disruption_features_zone_year_{config.scope}.parquet",
        od_year_features_path=processed_dir / f"disruption_features_od_year_{config.scope}.parquet",
        supervised_with_disruption_path=processed_dir / f"freight_mnet_supervised_edges_2018_2024_{config.scope}_with_disruption.parquet",
        disruption_manifest_path=processed_dir / f"disruption_feature_manifest_{config.scope}.json",
        run_config_path=experiment_dir / "run_config_build_disruption_features.json",
        qa_summary_path=tables_dir / "disruption_feature_qa_summary.csv",
        source_audit_path=tables_dir / "disruption_source_file_audit.csv",
    )

paths = build_paths(cfg)
for directory in [paths.processed_disruption_dir, paths.experiment_dir, paths.tables_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Supervised table:", paths.supervised_path)
print("Feature manifest:", paths.manifest_path)
print("Processed disruption directory:", paths.processed_disruption_dir)
print("Experiment directory:", paths.experiment_dir)

Supervised table: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Processed disruption directory: E:\NetworkOptimization\pythonProject1\Data\08_processed\disruption_features
Experiment directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\build_disruption_features_od_year_v1\east_plus_gulf


## 5. General utility functions

These helpers normalize FAF codes, county FIPS codes, years, and mixed object columns before Parquet writing. They also avoid optional pandas dependencies such as `tabulate`.

In [5]:
# ---------------------------------------------------------------------
# General utility functions
# ---------------------------------------------------------------------

LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
YEAR_CANDIDATES = ["year", "YEAR", "Year", "begin_year", "BEGIN_YEAR", "BeginYear"]


def ensure_file_exists(path: Path, description: str) -> None:
    """Raise a clear error if an expected file is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"{description} not found: {path}")


def normalize_faf_code(value: object) -> Optional[str]:
    """Normalize a FAF-zone identifier to a 3-character string."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    digits = re.findall(r"\d+", text)
    if not digits:
        return None
    return digits[-1].zfill(3)


def normalize_county_fips(value: object) -> Optional[str]:
    """Normalize a county FIPS code to a 5-character string."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    digits = re.findall(r"\d+", text)
    if not digits:
        return None
    return digits[-1].zfill(5)


def infer_year_from_series(series: pd.Series) -> pd.Series:
    """Infer calendar year from numeric, date-like, or year-month series."""
    if series.empty:
        return pd.Series(dtype="Int64")

    # Direct numeric year or YYYYMM format.
    numeric = pd.to_numeric(series, errors="coerce")
    years = pd.Series(pd.NA, index=series.index, dtype="Int64")

    direct_year_mask = numeric.between(1900, 2100, inclusive="both")
    years.loc[direct_year_mask] = numeric.loc[direct_year_mask].astype("Int64")

    yyyymm_mask = numeric.between(190001, 210012, inclusive="both")
    years.loc[yyyymm_mask] = (numeric.loc[yyyymm_mask] // 100).astype("Int64")

    # Date-like fallback.
    missing_mask = years.isna()
    if missing_mask.any():
        parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")
        years.loc[missing_mask] = parsed_dates.dt.year.astype("Int64")

    return years


def find_first_column(columns: Iterable[str], candidates: Sequence[str], required: bool = False) -> Optional[str]:
    """Find the first candidate column in a case-sensitive or case-insensitive way."""
    column_list = list(columns)
    column_set = set(column_list)
    for candidate in candidates:
        if candidate in column_set:
            return candidate

    lower_map = {str(column).lower(): column for column in column_list}
    for candidate in candidates:
        lowered = candidate.lower()
        if lowered in lower_map:
            return lower_map[lowered]

    if required:
        raise ValueError(f"None of the candidate columns were found: {candidates}")
    return None


def log_frame(name: str, frame: pd.DataFrame, max_rows: int = 5) -> None:
    """Print a compact dataframe preview without requiring tabulate."""
    print(f"\n{name}: shape={frame.shape}")
    if not frame.empty:
        print(frame.head(max_rows).to_string(index=False))


def normalize_dataframe_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """Convert object-like columns to strings so pyarrow can write reliably."""
    clean = df.copy()
    clean.columns = [str(column) for column in clean.columns]
    for column in clean.columns:
        if pd.api.types.is_object_dtype(clean[column]) or pd.api.types.is_string_dtype(clean[column]):
            clean[column] = clean[column].map(lambda x: pd.NA if x is None or (not isinstance(x, (list, dict, tuple, set)) and pd.isna(x)) else str(x)).astype("string")
    return clean


def safe_to_parquet(df: pd.DataFrame, path: Path) -> Path:
    """Write a dataframe to Parquet with robust object dtype handling."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    normalize_dataframe_for_parquet(df).to_parquet(path, index=False, engine="pyarrow")
    return path


def write_json(payload: dict, path: Path) -> Path:
    """Write a JSON file with a stable indentation style."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=str)
    return path

## 6. Load supervised table and feature manifest

The disruption table must align exactly with the supervised OD-year rows. This cell loads the supervised table, standardizes `faf_orig`, `faf_dest`, and `year`, and creates a stable `row_id` if needed.

In [6]:
# ---------------------------------------------------------------------
# Load supervised model-ready table and feature manifest
# ---------------------------------------------------------------------

ensure_file_exists(paths.supervised_path, "Supervised model-ready table")
ensure_file_exists(paths.manifest_path, "Feature manifest")

supervised_df = pd.read_parquet(paths.supervised_path).copy()
with paths.manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

manifest_feature_columns = manifest.get("feature_columns") or manifest.get("manifest_feature_columns")
if not manifest_feature_columns:
    raise ValueError("Feature manifest does not contain feature_columns.")

origin_col = find_first_column(supervised_df.columns, ["faf_orig", "orig_faf", "origin_faf", "origin", "o"], required=True)
dest_col = find_first_column(supervised_df.columns, ["faf_dest", "dest_faf", "destination_faf", "destination", "d"], required=True)
year_col = find_first_column(supervised_df.columns, ["year", "YEAR", "Year"], required=True)

if "row_id" not in supervised_df.columns:
    supervised_df["row_id"] = np.arange(len(supervised_df), dtype=np.int64)

supervised_df["faf_orig_str"] = supervised_df[origin_col].map(normalize_faf_code)
supervised_df["faf_dest_str"] = supervised_df[dest_col].map(normalize_faf_code)
supervised_df["year_int"] = pd.to_numeric(supervised_df[year_col], errors="coerce").astype("Int64")

required_label_missing = [column for column in LABEL_COLUMNS if column not in supervised_df.columns]
if required_label_missing:
    raise ValueError(f"Missing label columns: {required_label_missing}")

if supervised_df[["faf_orig_str", "faf_dest_str", "year_int"]].isna().any().any():
    raise ValueError("Origin, destination, or year normalization produced missing values.")

all_years = sorted(supervised_df["year_int"].dropna().astype(int).unique().tolist())
all_faf_zones = sorted(set(supervised_df["faf_orig_str"]) | set(supervised_df["faf_dest_str"]))

print("Supervised table shape:", supervised_df.shape)
print("Number of FAF zones in supervised table:", len(all_faf_zones))
print("Years:", all_years)
print("Manifest feature count:", len(manifest_feature_columns))
log_frame("Supervised preview", supervised_df[["row_id", "faf_orig_str", "faf_dest_str", "year_int"] + LABEL_COLUMNS].head())

Supervised table shape: (73972, 96)
Number of FAF zones in supervised table: 104
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
Manifest feature count: 64

Supervised preview: shape=(5, 7)
 row_id faf_orig_str faf_dest_str  year_int  truck_q25  truck_q50  truck_q75
      0          011          011      2018       1.00      37.02      71.28
      1          011          012      2018     248.18     943.27    1925.53
      2          011          019      2018      48.00     100.10     218.95
      3          011          050      2018     404.01    1010.39    1818.68
      4          011          091      2018    2579.38    3718.82    5127.33


## 7. Source file discovery and robust readers

The raw disruption data folders may contain Parquet, CSV, or Excel files. This cell provides a unified file discovery and reading layer with source auditing.

In [7]:
# ---------------------------------------------------------------------
# Source file discovery and robust table readers
# ---------------------------------------------------------------------

TABULAR_EXTENSIONS = {".parquet", ".pq", ".csv", ".txt", ".xlsx", ".xls"}


def discover_tabular_files(root: Path, max_files: Optional[int] = None, recursive: bool = True) -> List[Path]:
    """Discover tabular source files under a root directory."""
    root = Path(root)
    if not root.exists():
        return []
    iterator = root.rglob("*") if recursive else root.glob("*")
    files = sorted(path for path in iterator if path.is_file() and path.suffix.lower() in TABULAR_EXTENSIONS)
    if max_files is not None:
        files = files[:max_files]
    return files


def read_tabular_file(path: Path, nrows: Optional[int] = None) -> pd.DataFrame:
    """Read Parquet, CSV/TXT, or Excel files into a dataframe."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".csv", ".txt"}:
        return pd.read_csv(path, low_memory=False, nrows=nrows)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, nrows=nrows)
    raise ValueError(f"Unsupported file extension: {path}")


def build_source_audit() -> pd.DataFrame:
    """Create a source-file audit table for NOAA, STB, and border inputs."""
    rows = []
    for source_name, root, max_files in [
        ("noaa", cfg.noaa_root, cfg.max_noaa_files),
        ("stb", cfg.stb_root, cfg.max_stb_files),
        ("border", cfg.border_root, cfg.max_border_files),
    ]:
        files = discover_tabular_files(root, max_files=max_files, recursive=cfg.recursive_source_search)
        if not files:
            rows.append({"source": source_name, "path": str(root), "exists": Path(root).exists(), "file_count": 0, "used": False})
        else:
            for path in files:
                rows.append({"source": source_name, "path": str(path), "exists": True, "file_count": len(files), "used": True})
    return pd.DataFrame(rows)

source_audit = build_source_audit()
source_audit.to_csv(paths.source_audit_path, index=False)
log_frame("Source file audit", source_audit, max_rows=20)


Source file audit: shape=(2704, 5)
source                                                                                                                        path  exists  file_count  used
  noaa        E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\_metadata\noaa_storm_events_full_index_inventory.csv    True          65  True
  noaa    E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\_metadata\noaa_storm_events_link_inventory_2018_2024.csv    True          65  True
  noaa       E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\parquet\details\noaa_stormevents_details_2018.parquet    True          65  True
  noaa       E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\parquet\details\noaa_stormevents_details_2019.parquet    True          65  True
  noaa       E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\parquet\details\noaa_stormevents_details_2020.parquet    True          65  True
  noaa       E:\Networ

## 8. County-to-FAF crosswalk loading

NOAA storm events are usually county-based. This cell tries to find a processed county-to-FAF crosswalk and normalizes it to columns `county_fips` and `faf_zone_str`.

In [8]:
# ---------------------------------------------------------------------
# Load county-to-FAF crosswalk for NOAA county-level mapping
# ---------------------------------------------------------------------

COUNTY_CANDIDATES = [
    "county_fips", "COUNTY_FIPS", "ctfips", "CTFIPS", "fips", "FIPS",
    "GEOID", "geoid", "county_geoid", "COUNTYFP"
]
FAF_CANDIDATES = [
    "faf_zone", "FAF_ZONE", "faf", "FAF", "faf_id", "FAF_ID", "faf_zone_id",
    "zone", "ZONE", "FAFREGION", "faf_region"
]


def load_county_to_faf_crosswalk(root: Path) -> pd.DataFrame:
    """Find and load a county-to-FAF crosswalk under the configured root."""
    files = discover_tabular_files(root, max_files=None, recursive=True)
    candidates = [path for path in files if "faf" in path.name.lower() and ("county" in path.name.lower() or "ct" in path.name.lower())]
    candidates = candidates or files

    for path in candidates:
        try:
            frame = read_tabular_file(path)
            county_col = find_first_column(frame.columns, COUNTY_CANDIDATES)
            faf_col = find_first_column(frame.columns, FAF_CANDIDATES)
            if county_col is None or faf_col is None:
                continue
            out = frame[[county_col, faf_col]].copy()
            out.columns = ["county_fips", "faf_zone_str"]
            out["county_fips"] = out["county_fips"].map(normalize_county_fips)
            out["faf_zone_str"] = out["faf_zone_str"].map(normalize_faf_code)
            out = out.dropna(subset=["county_fips", "faf_zone_str"]).drop_duplicates()
            if not out.empty:
                print("Loaded county-to-FAF crosswalk:", path)
                return out
        except Exception as exc:
            print(f"Skipping crosswalk candidate {path}: {exc}")

    print("Warning: no usable county-to-FAF crosswalk found. NOAA zone-level features will fall back to global-year features.")
    return pd.DataFrame(columns=["county_fips", "faf_zone_str"])

county_to_faf = load_county_to_faf_crosswalk(cfg.crosswalk_root)
log_frame("County-to-FAF crosswalk", county_to_faf)
print("Unique counties:", county_to_faf["county_fips"].nunique() if not county_to_faf.empty else 0)
print("Unique FAF zones:", county_to_faf["faf_zone_str"].nunique() if not county_to_faf.empty else 0)

Loaded county-to-FAF crosswalk: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf.csv

County-to-FAF crosswalk: shape=(3144, 2)
county_fips faf_zone_str
      01001          019
      01003          012
      01005          019
      01007          011
      01009          011
Unique counties: 3144
Unique FAF zones: 132


## 9. NOAA storm-event feature engineering

This cell converts county-level storm-event records into FAF zone-year weather exposure features. It supports common NOAA Storm Events schemas, including `BEGIN_YEARMONTH`, `STATE_FIPS`, `CZ_FIPS`, `EVENT_TYPE`, damage columns, and duration columns.

If the county mapping fails, the notebook still creates global weather-year features and transparent missing-source flags.

In [9]:
# ---------------------------------------------------------------------
# NOAA storm-event feature engineering
# ---------------------------------------------------------------------

NOAA_YEAR_CANDIDATES = ["YEAR", "year", "BEGIN_YEAR", "begin_year", "BEGIN_YEARMONTH", "begin_yearmonth", "BEGIN_DATE_TIME", "begin_date_time"]
NOAA_EVENT_TYPE_CANDIDATES = ["EVENT_TYPE", "event_type", "Event Type", "EVENT", "event"]
NOAA_DAMAGE_PROPERTY_CANDIDATES = ["DAMAGE_PROPERTY", "damage_property", "Property Damage", "property_damage"]
NOAA_DAMAGE_CROPS_CANDIDATES = ["DAMAGE_CROPS", "damage_crops", "Crop Damage", "crop_damage"]
NOAA_INJURY_CANDIDATES = ["INJURIES_DIRECT", "injuries_direct", "INJURIES_INDIRECT", "injuries_indirect"]
NOAA_FATALITY_CANDIDATES = ["DEATHS_DIRECT", "deaths_direct", "DEATHS_INDIRECT", "deaths_indirect"]
NOAA_DURATION_CANDIDATES = ["DURATION_HOURS", "duration_hours", "EPISODE_DURATION", "episode_duration"]


def parse_noaa_damage(value: object) -> float:
    """Parse NOAA compact damage strings such as 10K, 2.5M, or 1B."""
    if value is None:
        return 0.0
    try:
        if pd.isna(value):
            return 0.0
    except (TypeError, ValueError):
        pass
    text = str(value).strip().upper().replace(",", "")
    if not text or text in {"0", "0.0", "NA", "NAN", "NONE"}:
        return 0.0
    match = re.match(r"^([-+]?\d*\.?\d+)([KMB]?)$", text)
    if not match:
        numeric = pd.to_numeric(text, errors="coerce")
        return float(numeric) if pd.notna(numeric) else 0.0
    base = float(match.group(1))
    suffix = match.group(2)
    multiplier = {"": 1.0, "K": 1_000.0, "M": 1_000_000.0, "B": 1_000_000_000.0}[suffix]
    return base * multiplier


def classify_weather_event(event_type: object) -> Dict[str, int]:
    """Create coarse event category indicators from NOAA EVENT_TYPE."""
    text = str(event_type).strip().lower() if event_type is not None else ""
    return {
        "weather_flood_count": int(any(token in text for token in ["flood", "flash flood", "coastal flood"])),
        "weather_winter_count": int(any(token in text for token in ["winter", "snow", "ice", "blizzard", "freeze", "frost", "sleet"])),
        "weather_tropical_count": int(any(token in text for token in ["hurricane", "tropical", "storm surge"])),
        "weather_convective_count": int(any(token in text for token in ["thunderstorm", "tornado", "lightning"])),
        "weather_wind_hail_count": int(any(token in text for token in ["wind", "hail", "dust"])),
        "weather_heat_cold_count": int(any(token in text for token in ["heat", "cold", "wind chill", "hypothermia"])),
        "weather_wildfire_count": int(any(token in text for token in ["wildfire", "fire"])),
    }


def infer_noaa_county_fips(frame: pd.DataFrame) -> pd.Series:
    """Infer county FIPS from common NOAA schema columns."""
    direct_col = find_first_column(frame.columns, COUNTY_CANDIDATES)
    if direct_col is not None:
        return frame[direct_col].map(normalize_county_fips)

    state_col = find_first_column(frame.columns, ["STATE_FIPS", "state_fips", "STATEFP", "statefp"])
    cz_col = find_first_column(frame.columns, ["CZ_FIPS", "cz_fips", "CZ_FIPS_NUM", "cz_fips_num", "COUNTYFP", "countyfp"])
    if state_col is not None and cz_col is not None:
        state = frame[state_col].map(lambda x: None if pd.isna(x) else str(int(float(x))).zfill(2) if str(x).replace('.', '', 1).isdigit() else str(x).zfill(2))
        county = frame[cz_col].map(lambda x: None if pd.isna(x) else str(int(float(x))).zfill(3) if str(x).replace('.', '', 1).isdigit() else str(x).zfill(3))
        return (state.fillna("") + county.fillna("")).map(normalize_county_fips)

    return pd.Series(pd.NA, index=frame.index, dtype="string")


def process_noaa_file(path: Path, county_crosswalk: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process one NOAA source file into zone-year and global-year partial aggregates."""
    frame = read_tabular_file(path)
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame()

    year_col = find_first_column(frame.columns, NOAA_YEAR_CANDIDATES)
    if year_col is None:
        return pd.DataFrame(), pd.DataFrame()

    out = pd.DataFrame(index=frame.index)
    out["year_int"] = infer_year_from_series(frame[year_col])
    out["county_fips"] = infer_noaa_county_fips(frame)

    event_col = find_first_column(frame.columns, NOAA_EVENT_TYPE_CANDIDATES)
    event_values = frame[event_col] if event_col is not None else pd.Series("unknown", index=frame.index)
    out["weather_event_count"] = 1.0

    category_rows = [classify_weather_event(value) for value in event_values]
    category_df = pd.DataFrame(category_rows, index=frame.index)
    out = pd.concat([out, category_df], axis=1)

    property_col = find_first_column(frame.columns, NOAA_DAMAGE_PROPERTY_CANDIDATES)
    crops_col = find_first_column(frame.columns, NOAA_DAMAGE_CROPS_CANDIDATES)
    property_damage = frame[property_col].map(parse_noaa_damage) if property_col is not None else pd.Series(0.0, index=frame.index)
    crop_damage = frame[crops_col].map(parse_noaa_damage) if crops_col is not None else pd.Series(0.0, index=frame.index)
    out["weather_damage_total_usd"] = property_damage.astype(float) + crop_damage.astype(float)

    injury_cols = [column for column in NOAA_INJURY_CANDIDATES if column in frame.columns]
    fatality_cols = [column for column in NOAA_FATALITY_CANDIDATES if column in frame.columns]
    out["weather_injury_count"] = frame[injury_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) if injury_cols else 0.0
    out["weather_fatality_count"] = frame[fatality_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) if fatality_cols else 0.0

    duration_col = find_first_column(frame.columns, NOAA_DURATION_CANDIDATES)
    out["weather_duration_hours"] = pd.to_numeric(frame[duration_col], errors="coerce").fillna(0.0) if duration_col is not None else 0.0

    out = out.dropna(subset=["year_int"])
    out["year_int"] = out["year_int"].astype(int)
    out = out[out["year_int"].isin(all_years)].copy()
    if out.empty:
        return pd.DataFrame(), pd.DataFrame()

    numeric_columns = [column for column in out.columns if column not in {"year_int", "county_fips"}]
    global_year = out.groupby("year_int", as_index=False)[numeric_columns].sum()

    if county_crosswalk.empty:
        return pd.DataFrame(), global_year

    mapped = out.merge(county_crosswalk, on="county_fips", how="left")
    mapped = mapped.dropna(subset=["faf_zone_str"])
    if mapped.empty:
        return pd.DataFrame(), global_year

    zone_year = mapped.groupby(["faf_zone_str", "year_int"], as_index=False)[numeric_columns].sum()
    return zone_year, global_year


def build_noaa_features() -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Build NOAA FAF-zone-year and global-year weather features."""
    files = discover_tabular_files(cfg.noaa_root, max_files=cfg.max_noaa_files, recursive=cfg.recursive_source_search)
    qa = {"source": "noaa", "n_files": len(files), "status": "ok" if files else "missing"}
    zone_parts, global_parts = [], []

    for idx, path in enumerate(files, start=1):
        try:
            zone_part, global_part = process_noaa_file(path, county_to_faf)
            if not zone_part.empty:
                zone_parts.append(zone_part)
            if not global_part.empty:
                global_parts.append(global_part)
            print(f"[NOAA] processed {idx}/{len(files)}: {path.name}")
        except Exception as exc:
            print(f"[NOAA] skipped {path}: {exc}")

    numeric_feature_columns = [
        "weather_event_count",
        "weather_flood_count",
        "weather_winter_count",
        "weather_tropical_count",
        "weather_convective_count",
        "weather_wind_hail_count",
        "weather_heat_cold_count",
        "weather_wildfire_count",
        "weather_damage_total_usd",
        "weather_injury_count",
        "weather_fatality_count",
        "weather_duration_hours",
    ]

    if zone_parts:
        zone = pd.concat(zone_parts, ignore_index=True)
        zone = zone.groupby(["faf_zone_str", "year_int"], as_index=False)[numeric_feature_columns].sum()
    else:
        zone = pd.DataFrame(columns=["faf_zone_str", "year_int"] + numeric_feature_columns)

    if global_parts:
        global_year = pd.concat(global_parts, ignore_index=True)
        global_year = global_year.groupby("year_int", as_index=False)[numeric_feature_columns].sum()
        global_year = global_year.rename(columns={column: f"weather_global_{column.replace('weather_', '')}" for column in numeric_feature_columns})
    else:
        global_year = pd.DataFrame({"year_int": all_years})
        for column in numeric_feature_columns:
            global_year[f"weather_global_{column.replace('weather_', '')}"] = 0.0

    for column in numeric_feature_columns:
        if column not in zone.columns:
            zone[column] = 0.0

    zone["weather_log1p_damage_total_usd"] = np.log1p(pd.to_numeric(zone["weather_damage_total_usd"], errors="coerce").fillna(0.0)) if not zone.empty else []
    global_damage_col = "weather_global_damage_total_usd"
    if global_damage_col in global_year.columns:
        global_year["weather_global_log1p_damage_total_usd"] = np.log1p(pd.to_numeric(global_year[global_damage_col], errors="coerce").fillna(0.0))

    return zone, global_year, qa

noaa_zone_year, noaa_global_year, noaa_qa = build_noaa_features()
log_frame("NOAA zone-year features", noaa_zone_year)
log_frame("NOAA global-year features", noaa_global_year)

[NOAA] processed 1/65: noaa_storm_events_full_index_inventory.csv
[NOAA] processed 2/65: noaa_storm_events_link_inventory_2018_2024.csv
[NOAA] processed 3/65: noaa_stormevents_details_2018.parquet
[NOAA] processed 4/65: noaa_stormevents_details_2019.parquet
[NOAA] processed 5/65: noaa_stormevents_details_2020.parquet
[NOAA] processed 6/65: noaa_stormevents_details_2021.parquet
[NOAA] processed 7/65: noaa_stormevents_details_2022.parquet
[NOAA] processed 8/65: noaa_stormevents_details_2023.parquet
[NOAA] processed 9/65: noaa_stormevents_details_2024.parquet
[NOAA] processed 10/65: noaa_stormevents_fatalities_2018.parquet
[NOAA] processed 11/65: noaa_stormevents_fatalities_2019.parquet
[NOAA] processed 12/65: noaa_stormevents_fatalities_2020.parquet
[NOAA] processed 13/65: noaa_stormevents_fatalities_2021.parquet
[NOAA] processed 14/65: noaa_stormevents_fatalities_2022.parquet
[NOAA] processed 15/65: noaa_stormevents_fatalities_2023.parquet
[NOAA] processed 16/65: noaa_stormevents_fatali

## 10. STB rail service stress feature engineering

STB files are often national, carrier-level, weekly, or monthly. This cell produces annual rail stress features. If a file contains a FAF-zone column, it also produces FAF zone-year features; otherwise it produces global year features that will be attached to every OD-year row.

In [10]:
# ---------------------------------------------------------------------
# STB rail service stress feature engineering
# ---------------------------------------------------------------------

RAIL_KEYWORDS = {
    "velocity": ["velocity", "speed"],
    "dwell": ["dwell"],
    "trains_held": ["held", "trainsheld", "train_held"],
    "cars_online": ["cars", "online", "carsonline"],
    "terminal": ["terminal"],
}


def infer_year_from_any_columns(frame: pd.DataFrame) -> pd.Series:
    """Infer year using explicit year/date columns or datetime-like columns."""
    year_col = find_first_column(frame.columns, YEAR_CANDIDATES)
    if year_col is not None:
        return infer_year_from_series(frame[year_col])

    for column in frame.columns:
        name = str(column).lower()
        if any(token in name for token in ["date", "week", "month"]):
            years = infer_year_from_series(frame[column])
            if years.notna().sum() > 0:
                return years

    return pd.Series(pd.NA, index=frame.index, dtype="Int64")


def infer_faf_zone_series(frame: pd.DataFrame) -> pd.Series:
    """Infer a FAF-zone column if one exists in a rail/border source table."""
    faf_col = find_first_column(frame.columns, FAF_CANDIDATES)
    if faf_col is None:
        return pd.Series(pd.NA, index=frame.index, dtype="string")
    return frame[faf_col].map(normalize_faf_code)


def find_numeric_columns_by_keywords(frame: pd.DataFrame, keywords: Sequence[str]) -> List[str]:
    """Find numeric columns whose names contain any keyword."""
    out = []
    for column in frame.columns:
        normalized_name = re.sub(r"[^a-z0-9]+", "", str(column).lower())
        if any(keyword in normalized_name for keyword in keywords):
            numeric = pd.to_numeric(frame[column], errors="coerce")
            if numeric.notna().sum() > 0:
                out.append(column)
    return out


def process_stb_file(path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process one STB rail service file into optional zone-year and global-year features."""
    frame = read_tabular_file(path)
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame()

    years = infer_year_from_any_columns(frame)
    valid = years.notna() & years.astype("Int64").isin(all_years)
    if valid.sum() == 0:
        return pd.DataFrame(), pd.DataFrame()

    out = pd.DataFrame({"year_int": years.loc[valid].astype(int)})
    out["faf_zone_str"] = infer_faf_zone_series(frame).loc[valid].values

    used_feature_cols = []
    for group_name, keywords in RAIL_KEYWORDS.items():
        cols = find_numeric_columns_by_keywords(frame, keywords)
        if cols:
            values = frame.loc[valid, cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
            out[f"rail_{group_name}_mean"] = values.fillna(0.0).astype(float)
            used_feature_cols.append(f"rail_{group_name}_mean")

    if not used_feature_cols:
        numeric_cols = [column for column in frame.columns if pd.to_numeric(frame[column], errors="coerce").notna().sum() > 0]
        numeric_cols = [column for column in numeric_cols if column not in {"year", "YEAR", "Year"}]
        if numeric_cols:
            out["rail_numeric_mean"] = frame.loc[valid, numeric_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1).fillna(0.0)
            used_feature_cols.append("rail_numeric_mean")
        else:
            return pd.DataFrame(), pd.DataFrame()

    global_year = out.groupby("year_int", as_index=False)[used_feature_cols].mean()

    zone = out.dropna(subset=["faf_zone_str"]).copy()
    if not zone.empty:
        zone_year = zone.groupby(["faf_zone_str", "year_int"], as_index=False)[used_feature_cols].mean()
    else:
        zone_year = pd.DataFrame()

    return zone_year, global_year


def build_stb_features() -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Build STB rail-service stress features."""
    files = discover_tabular_files(cfg.stb_root, max_files=cfg.max_stb_files, recursive=cfg.recursive_source_search)
    qa = {"source": "stb", "n_files": len(files), "status": "ok" if files else "missing"}
    zone_parts, global_parts = [], []

    for idx, path in enumerate(files, start=1):
        try:
            zone_part, global_part = process_stb_file(path)
            if not zone_part.empty:
                zone_parts.append(zone_part)
            if not global_part.empty:
                global_parts.append(global_part)
            print(f"[STB] processed {idx}/{len(files)}: {path.name}")
        except Exception as exc:
            print(f"[STB] skipped {path}: {exc}")

    if zone_parts:
        zone = pd.concat(zone_parts, ignore_index=True)
        numeric_cols = [column for column in zone.columns if column not in {"faf_zone_str", "year_int"}]
        zone = zone.groupby(["faf_zone_str", "year_int"], as_index=False)[numeric_cols].mean()
    else:
        zone = pd.DataFrame(columns=["faf_zone_str", "year_int"])

    if global_parts:
        global_year = pd.concat(global_parts, ignore_index=True)
        numeric_cols = [column for column in global_year.columns if column != "year_int"]
        global_year = global_year.groupby("year_int", as_index=False)[numeric_cols].mean()
    else:
        global_year = pd.DataFrame({"year_int": all_years})

    # Ensure standard columns exist.
    for col in ["rail_velocity_mean", "rail_dwell_mean", "rail_trains_held_mean", "rail_cars_online_mean", "rail_terminal_mean", "rail_numeric_mean"]:
        if col not in global_year.columns:
            global_year[col] = 0.0
        if col not in zone.columns:
            zone[col] = 0.0

    # Construct a simple annual rail stress index. High dwell/held is bad; high velocity is good.
    def add_stress_index(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        for col in ["rail_velocity_mean", "rail_dwell_mean", "rail_trains_held_mean", "rail_cars_online_mean"]:
            values = pd.to_numeric(out[col], errors="coerce").fillna(0.0)
            std = values.std(ddof=0)
            out[f"{col}_z"] = 0.0 if std == 0 or np.isnan(std) else (values - values.mean()) / std
        out["rail_stress_index"] = (
            out["rail_dwell_mean_z"]
            + out["rail_trains_held_mean_z"]
            + 0.25 * out["rail_cars_online_mean_z"]
            - out["rail_velocity_mean_z"]
        )
        return out

    global_year = add_stress_index(global_year)
    if not zone.empty:
        zone = add_stress_index(zone)

    global_year = global_year.rename(columns={col: f"rail_global_{col.replace('rail_', '')}" for col in global_year.columns if col != "year_int"})
    zone = zone.rename(columns={col: f"rail_zone_{col.replace('rail_', '')}" for col in zone.columns if col not in {"faf_zone_str", "year_int"}})

    return zone, global_year, qa

stb_zone_year, stb_global_year, stb_qa = build_stb_features()
log_frame("STB zone-year features", stb_zone_year)
log_frame("STB global-year features", stb_global_year)

[STB] processed 1/2619: stb_rail_service_link_inventory.csv
[STB] processed 2/2619: stb_rail_service_selected_download_links.csv
[STB] processed 3/2619: stb_rail_service_xlsx_sheet_preview.csv
[STB] processed 4/2619: 6104cf3ad4_STB-1145-BNSF.csv
[STB] processed 5/2619: 1189020d6c_STB-1145-CPKC.csv
[STB] processed 6/2619: f87b34edda_STB-1145-CSXT.csv
[STB] processed 7/2619: 406f4ee39a_STB-1145-ALL-CARRIERS.csv
[STB] processed 8/2619: 72a4e4c9d0_STB-1145-NS.csv
[STB] processed 9/2619: 966c50a9f5_STB-1145-GTC.csv
[STB] processed 10/2619: 9c19c4c693_STB-1145-UP.csv
[STB] processed 11/2619: 2026-05-13_f93c394325_EP724 Consolidated Data through 2026-05-13.xlsx
[STB] processed 12/2619: 2020-01-01_f28e2613c0_bnsf_data_2020-01-01.xlsx
[STB] processed 13/2619: 2020-01-08_e0ba864c3b_bnsf_data_2020-01-08.xlsx
[STB] processed 14/2619: 2020-01-15_4913e9f1ee_bnsf_data_2020-01-15.xlsx
[STB] processed 15/2619: 2020-01-22_829e25195f_bnsf_data_2020-01-22.xlsx
[STB] processed 16/2619: 2020-01-29_c804dde47

C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")


[STB] processed 1524/2619: 2023-05-31_0aa25e64cd_Chicago Data 2023-05-31.xlsx
[STB] processed 1525/2619: 2023-06-07_783bdbfcb1_Chicago Data 2023-06-07.xlsx
[STB] processed 1526/2619: 2023-06-14_063f00ee1b_Chicago Data 2023-06-14.xlsx
[STB] processed 1527/2619: 2023-06-21_626328500f_Chicago Data 2023-06-21.xlsx
[STB] processed 1528/2619: 2023-06-28_8b309bff15_Chicago Data 2023-06-28.xlsx
[STB] processed 1529/2619: 2023-07-05_262cf3b4b1_Chicago Data 2023-07-05.xlsx
[STB] processed 1530/2619: 2023-07-12_633c912fcd_Chicago Data 2023-07-12.xlsx
[STB] processed 1531/2619: 2023-07-19_a1d7cad3fa_Chicago Data 2023-07-19.xlsx
[STB] processed 1532/2619: 2023-07-26_f1af1ba71d_Chicago Data 2023-07-26.xlsx
[STB] processed 1533/2619: 2023-08-02_d9842b12a7_Chicago Data 2023-08-02.xlsx
[STB] processed 1534/2619: 2023-08-09_db0e11d472_Chicago Data 2023-08-09.xlsx
[STB] processed 1535/2619: 2023-08-16_033023719e_Chicago Data 2023-08-16.xlsx
[STB] processed 1536/2619: 2023-08-23_fc0a825675_Chicago Data 20

E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


[STB] processed 1869/2619: 2023-08-23_cee70f14b9_KCS Data 2023-08-23.xlsx
[STB] processed 1870/2619: 2023-08-30_8982110093_KCS Data 2023-08-30.xlsx
[STB] processed 1871/2619: 2023-09-06_1256afaf53_KCS Data 2023-09-06.xlsx
[STB] processed 1872/2619: 2023-09-13_7a4b3eeaa3_KCS Data 2023-09-13.xlsx
[STB] processed 1873/2619: 2023-09-20_3235639db3_KCS Data 2023-09-20.xlsx
[STB] processed 1874/2619: 2023-09-27_71c331ed73_KCS Data 2023-09-27.xlsx
[STB] processed 1875/2619: 2023-10-04_424d9f1e1b_KCS Data 2023-10-04.xlsx
[STB] processed 1876/2619: 2023-10-11_298dbba346_KCS Data 2023-10-11.xlsx
[STB] processed 1877/2619: 2023-10-18_97ab7d8b7c_KCS Data 2023-10-18.xlsx
[STB] processed 1878/2619: 2023-10-25_390dba46d8_KCS Data 2023-10-25.xlsx
[STB] processed 1879/2619: 2023-11-01_114459938f_KCS Data 2023-11-01.xlsx
[STB] processed 1880/2619: 2023-11-08_ac6a3b253d_KCS Data 2023-11-08.xlsx
[STB] processed 1881/2619: 2023-11-15_ed5df6d719_KCS Data 2023-11-15.xlsx
[STB] processed 1882/2619: 2023-11-22_

## 11. Border gateway stress feature engineering

Border Crossing data is often port/month/measure/value based and does not always have a direct FAF-zone key. This cell creates annual border gateway stress features. If a FAF-zone field exists, it also creates zone-year border features; otherwise it creates global annual features.

In [11]:
# ---------------------------------------------------------------------
# Border gateway stress feature engineering
# ---------------------------------------------------------------------

BORDER_VALUE_CANDIDATES = ["Value", "VALUE", "value", "Count", "count", "Crossings", "crossings"]
BORDER_MEASURE_CANDIDATES = ["Measure", "MEASURE", "measure", "Border Crossing/Entry", "metric"]
BORDER_DATE_CANDIDATES = ["Date", "DATE", "date", "Month", "MONTH", "month", "Year", "YEAR", "year"]
BORDER_NAME_CANDIDATES = ["Border", "BORDER", "border"]


def infer_border_year(frame: pd.DataFrame) -> pd.Series:
    """Infer year from common border data columns."""
    for candidate in BORDER_DATE_CANDIDATES:
        if candidate in frame.columns:
            years = infer_year_from_series(frame[candidate])
            if years.notna().sum() > 0:
                return years
    return infer_year_from_any_columns(frame)


def process_border_file(path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process one border crossing source file."""
    frame = read_tabular_file(path)
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame()

    years = infer_border_year(frame)
    valid = years.notna() & years.astype("Int64").isin(all_years)
    if valid.sum() == 0:
        return pd.DataFrame(), pd.DataFrame()

    value_col = find_first_column(frame.columns, BORDER_VALUE_CANDIDATES)
    if value_col is None:
        numeric_cols = [column for column in frame.columns if pd.to_numeric(frame[column], errors="coerce").notna().sum() > 0]
        value = frame[numeric_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1) if numeric_cols else pd.Series(0.0, index=frame.index)
    else:
        value = pd.to_numeric(frame[value_col], errors="coerce").fillna(0.0)

    measure_col = find_first_column(frame.columns, BORDER_MEASURE_CANDIDATES)
    if measure_col is not None:
        measure_text = frame[measure_col].astype(str).str.lower()
        truck_mask = measure_text.str.contains("truck", na=False)
        if truck_mask.any():
            valid = valid & truck_mask

    if valid.sum() == 0:
        return pd.DataFrame(), pd.DataFrame()

    out = pd.DataFrame({"year_int": years.loc[valid].astype(int)})
    out["border_truck_crossings"] = value.loc[valid].astype(float).values
    out["faf_zone_str"] = infer_faf_zone_series(frame).loc[valid].values

    border_col = find_first_column(frame.columns, BORDER_NAME_CANDIDATES)
    if border_col is not None:
        border_text = frame.loc[valid, border_col].astype(str).str.lower()
        out["border_mexico_truck_crossings"] = np.where(border_text.str.contains("mexico", na=False), out["border_truck_crossings"], 0.0)
        out["border_canada_truck_crossings"] = np.where(border_text.str.contains("canada", na=False), out["border_truck_crossings"], 0.0)
    else:
        out["border_mexico_truck_crossings"] = 0.0
        out["border_canada_truck_crossings"] = 0.0

    global_year = out.groupby("year_int", as_index=False).agg(
        border_truck_crossings_total=("border_truck_crossings", "sum"),
        border_truck_crossings_mean=("border_truck_crossings", "mean"),
        border_truck_crossings_std=("border_truck_crossings", "std"),
        border_mexico_truck_crossings_total=("border_mexico_truck_crossings", "sum"),
        border_canada_truck_crossings_total=("border_canada_truck_crossings", "sum"),
    )
    global_year["border_truck_crossings_std"] = global_year["border_truck_crossings_std"].fillna(0.0)

    zone = out.dropna(subset=["faf_zone_str"]).copy()
    if not zone.empty:
        zone_year = zone.groupby(["faf_zone_str", "year_int"], as_index=False).agg(
            border_zone_truck_crossings_total=("border_truck_crossings", "sum"),
            border_zone_truck_crossings_mean=("border_truck_crossings", "mean"),
        )
    else:
        zone_year = pd.DataFrame()

    return zone_year, global_year


def build_border_features() -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Build border crossing annual stress features."""
    files = discover_tabular_files(cfg.border_root, max_files=cfg.max_border_files, recursive=cfg.recursive_source_search)
    qa = {"source": "border", "n_files": len(files), "status": "ok" if files else "missing"}
    zone_parts, global_parts = [], []

    for idx, path in enumerate(files, start=1):
        try:
            zone_part, global_part = process_border_file(path)
            if not zone_part.empty:
                zone_parts.append(zone_part)
            if not global_part.empty:
                global_parts.append(global_part)
            print(f"[Border] processed {idx}/{len(files)}: {path.name}")
        except Exception as exc:
            print(f"[Border] skipped {path}: {exc}")

    if zone_parts:
        zone = pd.concat(zone_parts, ignore_index=True)
        numeric_cols = [column for column in zone.columns if column not in {"faf_zone_str", "year_int"}]
        zone = zone.groupby(["faf_zone_str", "year_int"], as_index=False)[numeric_cols].sum()
    else:
        zone = pd.DataFrame(columns=["faf_zone_str", "year_int"])

    if global_parts:
        global_year = pd.concat(global_parts, ignore_index=True)
        numeric_cols = [column for column in global_year.columns if column != "year_int"]
        global_year = global_year.groupby("year_int", as_index=False)[numeric_cols].sum()
    else:
        global_year = pd.DataFrame({"year_int": all_years})

    for col in [
        "border_truck_crossings_total",
        "border_truck_crossings_mean",
        "border_truck_crossings_std",
        "border_mexico_truck_crossings_total",
        "border_canada_truck_crossings_total",
    ]:
        if col not in global_year.columns:
            global_year[col] = 0.0

    global_year["border_log1p_truck_crossings_total"] = np.log1p(pd.to_numeric(global_year["border_truck_crossings_total"], errors="coerce").fillna(0.0))
    global_year = global_year.rename(columns={col: f"border_global_{col.replace('border_', '')}" for col in global_year.columns if col != "year_int"})

    return zone, global_year, qa

border_zone_year, border_global_year, border_qa = build_border_features()
log_frame("Border zone-year features", border_zone_year)
log_frame("Border global-year features", border_global_year)

[Border] processed 1/20: border_crossing_entry_data_keg4-3bc2_columns.csv
[Border] processed 2/20: border_crossing_keg4-3bc2_chunk_00000_offset_0.csv
[Border] processed 3/20: border_crossing_keg4-3bc2_chunk_00001_offset_50000.csv
[Border] processed 4/20: border_crossing_keg4-3bc2_chunk_00002_offset_100000.csv
[Border] processed 5/20: border_crossing_keg4-3bc2_chunk_00003_offset_150000.csv
[Border] processed 6/20: border_crossing_keg4-3bc2_chunk_00004_offset_200000.csv
[Border] processed 7/20: border_crossing_keg4-3bc2_chunk_00005_offset_250000.csv
[Border] processed 8/20: border_crossing_keg4-3bc2_chunk_00000_offset_0.parquet
[Border] processed 9/20: border_crossing_keg4-3bc2_chunk_00001_offset_50000.parquet
[Border] processed 10/20: border_crossing_keg4-3bc2_chunk_00002_offset_100000.parquet
[Border] processed 11/20: border_crossing_keg4-3bc2_chunk_00003_offset_150000.parquet
[Border] processed 12/20: border_crossing_keg4-3bc2_chunk_00004_offset_200000.parquet
[Border] processed 13/20

C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")


[Border] processed 14/20: border_crossing_entry_data_keg4-3bc2_full.parquet
[Border] processed 15/20: border_crossing_entry_data_keg4-3bc2_preview_first50.csv
[Border] processed 16/20: border_crossing_freight_measures_2018_2024.csv
[Border] processed 17/20: border_crossing_freight_measures_2018_2024.parquet


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")
C:\U

[Border] processed 18/20: border_crossing_gateway_monthly_pivot_2018_2024.parquet
[Border] processed 19/20: border_crossing_measure_inventory.csv


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68940\2426071979.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(series.loc[missing_mask], errors="coerce")


[Border] processed 20/20: border_crossing_entry_data_keg4-3bc2_full.csv

Border zone-year features: shape=(0, 2)

Border global-year features: shape=(7, 7)
 year_int  border_global_truck_crossings_total  border_global_truck_crossings_mean  border_global_truck_crossings_std  border_global_mexico_truck_crossings_total  border_global_canada_truck_crossings_total  border_global_log1p_truck_crossings_total
     2018                           90214746.0                        77771.332759                      194899.839738                                  45252126.0                                  44962620.0                                  18.317703
     2019                           90034546.0                        78087.203816                      195269.979566                                  46162730.0                                  43871816.0                                  18.315704
     2020                           86247078.0                        74737.502600               

## 12. Build the zone-year disruption table

This cell merges NOAA, STB, and border features onto a complete FAF zone-year grid. Missing values are filled with zeros and explicit source-missing flags are added.

In [12]:
# ---------------------------------------------------------------------
# Build complete zone-year disruption table
# ---------------------------------------------------------------------

zone_year_grid = pd.MultiIndex.from_product([all_faf_zones, all_years], names=["faf_zone_str", "year_int"]).to_frame(index=False)
zone_year = zone_year_grid.copy()

# Merge zone-level features.
for partial, source_name in [
    (noaa_zone_year, "weather"),
    (stb_zone_year, "rail"),
    (border_zone_year, "border"),
]:
    if partial is not None and not partial.empty:
        zone_year = zone_year.merge(partial, on=["faf_zone_str", "year_int"], how="left")
    else:
        zone_year[f"{source_name}_zone_source_missing"] = 1.0

# Fill numeric missing values after zone-level merge.
for column in zone_year.columns:
    if column not in {"faf_zone_str", "year_int"}:
        zone_year[column] = pd.to_numeric(zone_year[column], errors="coerce").fillna(0.0)

# Add source-available flags.
zone_year["weather_zone_feature_available"] = float(not noaa_zone_year.empty)
zone_year["rail_zone_feature_available"] = float(not stb_zone_year.empty)
zone_year["border_zone_feature_available"] = float(not border_zone_year.empty)

safe_to_parquet(zone_year, paths.zone_year_features_path)
log_frame("Complete zone-year disruption features", zone_year)
print("Zone-year disruption feature table saved to:", paths.zone_year_features_path)


Complete zone-year disruption features: shape=(728, 20)
faf_zone_str  year_int  weather_event_count  weather_flood_count  weather_winter_count  weather_tropical_count  weather_convective_count  weather_wind_hail_count  weather_heat_cold_count  weather_wildfire_count  weather_damage_total_usd  weather_injury_count  weather_fatality_count  weather_duration_hours  weather_log1p_damage_total_usd  rail_zone_source_missing  border_zone_source_missing  weather_zone_feature_available  rail_zone_feature_available  border_zone_feature_available
         011      2018                125.0                  7.0                  21.0                     2.0                      67.0                     78.0                      8.0                     0.0                  602900.0                  16.0                     0.0                     0.0                       13.309508                       1.0                         1.0                             1.0                          0.0     

## 13. Build global year-level disruption table

STB and border features are often year-level or national-level. NOAA also has a global aggregation fallback. This cell merges all global-year features and prepares them for OD-year rows.

In [13]:
# ---------------------------------------------------------------------
# Build global annual disruption features
# ---------------------------------------------------------------------

global_year = pd.DataFrame({"year_int": all_years})
for partial in [noaa_global_year, stb_global_year, border_global_year]:
    if partial is not None and not partial.empty:
        global_year = global_year.merge(partial, on="year_int", how="left")

for column in global_year.columns:
    if column != "year_int":
        global_year[column] = pd.to_numeric(global_year[column], errors="coerce").fillna(0.0)

# Add transparent source flags.
global_year["disrupt_noaa_source_file_count"] = float(noaa_qa.get("n_files", 0))
global_year["disrupt_stb_source_file_count"] = float(stb_qa.get("n_files", 0))
global_year["disrupt_border_source_file_count"] = float(border_qa.get("n_files", 0))

global_year.to_csv(paths.tables_dir / "global_year_disruption_features.csv", index=False)
log_frame("Global year disruption features", global_year)


Global year disruption features: shape=(7, 34)
 year_int  weather_global_event_count  weather_global_flood_count  weather_global_winter_count  weather_global_tropical_count  weather_global_convective_count  weather_global_wind_hail_count  weather_global_heat_cold_count  weather_global_wildfire_count  weather_global_damage_total_usd  weather_global_injury_count  weather_global_fatality_count  weather_global_duration_hours  weather_global_log1p_damage_total_usd  rail_global_numeric_mean  rail_global_cars_online_mean  rail_global_velocity_mean  rail_global_dwell_mean  rail_global_trains_held_mean  rail_global_terminal_mean  rail_global_velocity_mean_z  rail_global_dwell_mean_z  rail_global_trains_held_mean_z  rail_global_cars_online_mean_z  rail_global_stress_index  border_global_truck_crossings_total  border_global_truck_crossings_mean  border_global_truck_crossings_std  border_global_mexico_truck_crossings_total  border_global_canada_truck_crossings_total  border_global_log1p_truck_cro

## 14. Build OD-year disruption features

This is the main output table for D-CQHGT. Each row aligns with one supervised OD-year row via `row_id`. The feature construction combines:

- origin-zone disruption exposure;
- destination-zone disruption exposure;
- mean, max, and absolute-difference OD summaries;
- global annual rail/border/weather stress features.

In [14]:
# ---------------------------------------------------------------------
# Build OD-year disruption features aligned to supervised rows
# ---------------------------------------------------------------------

ZONE_KEY_COLUMNS = {"faf_zone_str", "year_int"}
zone_feature_columns = [column for column in zone_year.columns if column not in ZONE_KEY_COLUMNS]

origin_zone = zone_year.rename(columns={"faf_zone_str": "faf_orig_str"})
origin_zone = origin_zone.rename(columns={column: f"disrupt_origin_{column}" for column in zone_feature_columns})

dest_zone = zone_year.rename(columns={"faf_zone_str": "faf_dest_str"})
dest_zone = dest_zone.rename(columns={column: f"disrupt_dest_{column}" for column in zone_feature_columns})

od_features = supervised_df[["row_id", "faf_orig_str", "faf_dest_str", "year_int"]].copy()
od_features = od_features.merge(origin_zone, on=["faf_orig_str", "year_int"], how="left")
od_features = od_features.merge(dest_zone, on=["faf_dest_str", "year_int"], how="left")
od_features = od_features.merge(global_year, on="year_int", how="left")

# Create origin/destination pair summaries for zone-level disruption columns.
for base_col in zone_feature_columns:
    origin_col_name = f"disrupt_origin_{base_col}"
    dest_col_name = f"disrupt_dest_{base_col}"
    if origin_col_name in od_features.columns and dest_col_name in od_features.columns:
        origin_values = pd.to_numeric(od_features[origin_col_name], errors="coerce").fillna(0.0)
        dest_values = pd.to_numeric(od_features[dest_col_name], errors="coerce").fillna(0.0)
        od_features[f"disrupt_mean_{base_col}"] = 0.5 * (origin_values + dest_values)
        od_features[f"disrupt_max_{base_col}"] = np.maximum(origin_values, dest_values)
        od_features[f"disrupt_absdiff_{base_col}"] = np.abs(origin_values - dest_values)

# Prefix global year-level features if they do not already have a clear disruption prefix.
rename_global = {}
for column in od_features.columns:
    if column in {"row_id", "faf_orig_str", "faf_dest_str", "year_int"}:
        continue
    if column.startswith(("weather_global_", "rail_global_", "border_global_", "disrupt_")):
        rename_global[column] = f"disrupt_{column}" if not column.startswith("disrupt_") else column
od_features = od_features.rename(columns=rename_global)

# Ensure every candidate feature is numeric and finite.
non_feature_cols = {"row_id", "faf_orig_str", "faf_dest_str", "year_int"}
disruption_feature_columns = [column for column in od_features.columns if column not in non_feature_cols]
for column in disruption_feature_columns:
    od_features[column] = pd.to_numeric(od_features[column], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Stable column ordering.
od_features = od_features[["row_id", "faf_orig_str", "faf_dest_str", "year_int"] + disruption_feature_columns]

if len(od_features) != len(supervised_df):
    raise ValueError("OD-year disruption feature table does not align with supervised table length.")

if od_features["row_id"].duplicated().any():
    raise ValueError("OD-year disruption feature table has duplicated row_id values.")

safe_to_parquet(od_features, paths.od_year_features_path)
log_frame("OD-year disruption features", od_features)
print("OD-year disruption feature table saved to:", paths.od_year_features_path)
print("Disruption feature count:", len(disruption_feature_columns))


OD-year disruption features: shape=(73972, 127)
 row_id faf_orig_str faf_dest_str  year_int  disrupt_origin_weather_event_count  disrupt_origin_weather_flood_count  disrupt_origin_weather_winter_count  disrupt_origin_weather_tropical_count  disrupt_origin_weather_convective_count  disrupt_origin_weather_wind_hail_count  disrupt_origin_weather_heat_cold_count  disrupt_origin_weather_wildfire_count  disrupt_origin_weather_damage_total_usd  disrupt_origin_weather_injury_count  disrupt_origin_weather_fatality_count  disrupt_origin_weather_duration_hours  disrupt_origin_weather_log1p_damage_total_usd  disrupt_origin_rail_zone_source_missing  disrupt_origin_border_zone_source_missing  disrupt_origin_weather_zone_feature_available  disrupt_origin_rail_zone_feature_available  disrupt_origin_border_zone_feature_available  disrupt_dest_weather_event_count  disrupt_dest_weather_flood_count  disrupt_dest_weather_winter_count  disrupt_dest_weather_tropical_count  disrupt_dest_weather_convective_co

## 15. Append disruption features to the supervised table

This cell creates a convenience model-ready table with disruption features appended. Downstream D-CQHGT notebooks can either read this table directly or read the separate OD-year disruption feature table.

In [15]:
# ---------------------------------------------------------------------
# Save supervised table with disruption features appended
# ---------------------------------------------------------------------

supervised_with_disruption = supervised_df.merge(
    od_features.drop(columns=["faf_orig_str", "faf_dest_str", "year_int"]),
    on="row_id",
    how="left",
    validate="one_to_one",
)

missing_any = supervised_with_disruption[disruption_feature_columns].isna().any().any()
if missing_any:
    raise ValueError("Merged supervised-with-disruption table contains missing disruption values.")

safe_to_parquet(supervised_with_disruption, paths.supervised_with_disruption_path)
print("Saved supervised table with disruption features:", paths.supervised_with_disruption_path)
print("Shape:", supervised_with_disruption.shape)

Saved supervised table with disruption features: E:\NetworkOptimization\pythonProject1\Data\08_processed\disruption_features\freight_mnet_supervised_edges_2018_2024_east_plus_gulf_with_disruption.parquet
Shape: (73972, 219)


## 16. Feature manifest and leakage guard

Disruption features must not include label columns or FMI aggregation diagnostics. This cell builds a manifest for downstream model notebooks and checks that every disruption column is numeric.

In [16]:
# ---------------------------------------------------------------------
# Disruption feature manifest and leakage guard
# ---------------------------------------------------------------------

LEAKAGE_TOKENS = ["truck_q25", "truck_q50", "truck_q75", "input_q", "label", "target"]
leaky_columns = [column for column in disruption_feature_columns if any(token in column.lower() for token in LEAKAGE_TOKENS)]
if leaky_columns:
    raise ValueError(f"Potential label leakage in disruption features: {leaky_columns}")

non_numeric_disruption = [column for column in disruption_feature_columns if not pd.api.types.is_numeric_dtype(od_features[column])]
if non_numeric_disruption:
    raise TypeError(f"Non-numeric disruption features found: {non_numeric_disruption}")

feature_manifest = {
    "scope": cfg.scope,
    "created_at_unix": time.time(),
    "supervised_path": str(paths.supervised_path),
    "od_year_disruption_features_path": str(paths.od_year_features_path),
    "zone_year_disruption_features_path": str(paths.zone_year_features_path),
    "supervised_with_disruption_path": str(paths.supervised_with_disruption_path),
    "base_manifest_feature_columns": manifest_feature_columns,
    "disruption_feature_columns": disruption_feature_columns,
    "label_columns": LABEL_COLUMNS,
    "source_counts": {
        "noaa_files": int(noaa_qa.get("n_files", 0)),
        "stb_files": int(stb_qa.get("n_files", 0)),
        "border_files": int(border_qa.get("n_files", 0)),
    },
}
write_json(feature_manifest, paths.disruption_manifest_path)
print("Saved disruption feature manifest:", paths.disruption_manifest_path)
print("Disruption feature count:", len(disruption_feature_columns))

Saved disruption feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\disruption_features\disruption_feature_manifest_east_plus_gulf.json
Disruption feature count: 123


## 17. QA summaries

This cell writes compact QA tables that show feature coverage, nonzero rates, and split-level means. These files are useful for deciding whether the disruption signal is strong enough to move into a disruption-gated D-CQHGT experiment.

In [17]:
# ---------------------------------------------------------------------
# QA summaries
# ---------------------------------------------------------------------

qa_rows = []
for column in disruption_feature_columns:
    values = pd.to_numeric(od_features[column], errors="coerce").fillna(0.0)
    qa_rows.append({
        "feature": column,
        "mean": float(values.mean()),
        "std": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max()),
        "nonzero_rate": float((values != 0).mean()),
        "missing_rate": float(pd.isna(od_features[column]).mean()),
    })
qa_summary = pd.DataFrame(qa_rows).sort_values(["nonzero_rate", "std"], ascending=False)
qa_summary.to_csv(paths.qa_summary_path, index=False)

# Split-level means for a quick sanity check.
split_cols = [column for column in ["temporal_split", "cold_split"] if column in supervised_df.columns]
if split_cols:
    split_base = supervised_df[["row_id"] + split_cols].merge(od_features, on="row_id", how="left")
    for split_col in split_cols:
        split_summary = split_base.groupby(split_col)[disruption_feature_columns].mean(numeric_only=True).reset_index()
        split_summary.to_csv(paths.tables_dir / f"disruption_feature_means_by_{split_col}.csv", index=False)

log_frame("Disruption feature QA summary", qa_summary, max_rows=20)


Disruption feature QA summary: shape=(123, 7)
                                           feature         mean          std           min          max  nonzero_rate  missing_rate
           disrupt_weather_global_damage_total_usd 2.197017e+10 9.854678e+09  7.641288e+09 4.271832e+10           1.0           0.0
       disrupt_border_global_truck_crossings_total 9.236491e+07 3.530480e+06  8.624708e+07 9.744791e+07           1.0           0.0
disrupt_border_global_mexico_truck_crossings_total 4.945226e+07 3.468456e+06  4.525213e+07 5.450871e+07           1.0           0.0
disrupt_border_global_canada_truck_crossings_total 4.291265e+07 1.240893e+06  4.055839e+07 4.496262e+07           1.0           0.0
         disrupt_border_global_truck_crossings_std 2.133986e+05 2.731760e+04  1.860184e+05 2.725329e+05           1.0           0.0
        disrupt_border_global_truck_crossings_mean 8.307366e+04 9.064955e+03  7.473750e+04 1.041404e+05           1.0           0.0
                disrupt_weath

## 18. Source QA and missing-source interpretation

A missing source folder does not invalidate the notebook run, but it means the corresponding feature block is a transparent zero-valued fallback. This cell records source coverage and creates an interpretation table for downstream experiments.

In [18]:
# ---------------------------------------------------------------------
# Source QA and interpretation flags
# ---------------------------------------------------------------------

source_qa = pd.DataFrame([noaa_qa, stb_qa, border_qa])
source_qa.to_csv(paths.tables_dir / "disruption_source_qa.csv", index=False)

interpretation_rows = []
for source_name, qa in [("NOAA weather", noaa_qa), ("STB rail stress", stb_qa), ("Border gateway", border_qa)]:
    n_files = int(qa.get("n_files", 0))
    interpretation_rows.append({
        "source_block": source_name,
        "n_files_used_or_seen": n_files,
        "status": "active" if n_files > 0 else "fallback_zero",
        "interpretation": "Feature block has source files." if n_files > 0 else "No source files found; downstream disruption ablation for this block should be interpreted as a fallback control.",
    })
interpretation_table = pd.DataFrame(interpretation_rows)
interpretation_table.to_csv(paths.tables_dir / "disruption_source_interpretation.csv", index=False)
log_frame("Disruption source interpretation", interpretation_table)


Disruption source interpretation: shape=(3, 4)
   source_block  n_files_used_or_seen status                  interpretation
   NOAA weather                    65 active Feature block has source files.
STB rail stress                  2619 active Feature block has source files.
 Border gateway                    20 active Feature block has source files.


## 19. Run configuration and artifact manifest

This cell saves all paths and package versions for reproducibility.

In [19]:
# ---------------------------------------------------------------------
# Save run configuration and artifact manifest
# ---------------------------------------------------------------------

run_config = {
    "notebook": "Build_Disruption_Features_ODYear",
    "config": {key: str(value) for key, value in asdict(cfg).items()},
    "paths": {key: str(value) for key, value in asdict(paths).items()},
    "dataset": {
        "n_supervised_rows": int(len(supervised_df)),
        "n_faf_zones": int(len(all_faf_zones)),
        "years": [int(year) for year in all_years],
        "n_disruption_features": int(len(disruption_feature_columns)),
    },
    "source_qa": {
        "noaa": noaa_qa,
        "stb": stb_qa,
        "border": border_qa,
    },
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
write_json(run_config, paths.run_config_path)

artifact_manifest = {
    "zone_year_features": str(paths.zone_year_features_path),
    "od_year_features": str(paths.od_year_features_path),
    "supervised_with_disruption": str(paths.supervised_with_disruption_path),
    "disruption_feature_manifest": str(paths.disruption_manifest_path),
    "qa_summary": str(paths.qa_summary_path),
    "source_audit": str(paths.source_audit_path),
    "source_qa": str(paths.tables_dir / "disruption_source_qa.csv"),
    "source_interpretation": str(paths.tables_dir / "disruption_source_interpretation.csv"),
    "run_config": str(paths.run_config_path),
}
write_json(artifact_manifest, paths.experiment_dir / "analysis_artifact_manifest_build_disruption_features.json")

print("Saved disruption feature artifacts:")
print(json.dumps(artifact_manifest, indent=2))

Saved disruption feature artifacts:
{
  "zone_year_features": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\disruption_features\\disruption_features_zone_year_east_plus_gulf.parquet",
  "od_year_features": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\disruption_features\\disruption_features_od_year_east_plus_gulf.parquet",
  "supervised_with_disruption": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\disruption_features\\freight_mnet_supervised_edges_2018_2024_east_plus_gulf_with_disruption.parquet",
  "disruption_feature_manifest": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\disruption_features\\disruption_feature_manifest_east_plus_gulf.json",
  "qa_summary": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\build_disruption_features_od_year_v1\\east_plus_gulf\\tables\\disruption_feature_qa_summary.csv",
  "source_audit": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\build_disruption_fea

## 20. Loading test for downstream notebooks

The final cell reloads the three core artifacts and verifies row alignment. If this passes, the outputs are ready for `DCQHGT_Disruption_ColdOD.ipynb`.

In [20]:
# ---------------------------------------------------------------------
# Loading test for downstream D-CQHGT disruption experiments
# ---------------------------------------------------------------------

loaded_zone_year = pd.read_parquet(paths.zone_year_features_path)
loaded_od_year = pd.read_parquet(paths.od_year_features_path)
loaded_supervised_disruption = pd.read_parquet(paths.supervised_with_disruption_path)

with paths.disruption_manifest_path.open("r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

print("Loaded zone-year disruption table:", loaded_zone_year.shape)
print("Loaded OD-year disruption table:", loaded_od_year.shape)
print("Loaded supervised-with-disruption table:", loaded_supervised_disruption.shape)
print("Loaded disruption manifest keys:", sorted(loaded_manifest.keys()))

if len(loaded_od_year) != len(supervised_df):
    raise ValueError("Reloaded OD-year disruption table row count does not match supervised table.")

if loaded_od_year["row_id"].nunique() != len(supervised_df):
    raise ValueError("Reloaded OD-year disruption table does not have one unique row_id per supervised row.")

for column in loaded_manifest["disruption_feature_columns"]:
    if column not in loaded_od_year.columns:
        raise ValueError(f"Manifest feature missing from OD-year table: {column}")

print("Loading test completed successfully.")
print("Next notebook: DCQHGT_Disruption_ColdOD.ipynb")

Loaded zone-year disruption table: (728, 20)
Loaded OD-year disruption table: (73972, 127)
Loaded supervised-with-disruption table: (73972, 219)
Loaded disruption manifest keys: ['base_manifest_feature_columns', 'created_at_unix', 'disruption_feature_columns', 'label_columns', 'od_year_disruption_features_path', 'scope', 'source_counts', 'supervised_path', 'supervised_with_disruption_path', 'zone_year_disruption_features_path']
Loading test completed successfully.
Next notebook: DCQHGT_Disruption_ColdOD.ipynb
